In [1]:
import gymnasium as gym
from gymnasium import spaces
import cv2
import numpy as np
import time
import threading
import asyncio
import json
import uuid
from collections import deque
from typing import Optional, Tuple, Dict, Any
import sys
import concurrent.futures

# Your existing imports
import gi
from pipewire_capture import PortalCapture, CaptureStream, is_available
import websockets
from gi.repository import Gst, GLib

# Initialize GStreamer (once per process)
Gst.init(None)

/tmp/ipykernel_210854/1968732733.py:19: PyGIWarning: Gst was imported without specifying a version first. Use gi.require_version('Gst', '1.0') before import to ensure that the right version gets loaded.
  from gi.repository import Gst, GLib


[]

----

# Screen Capture

In [2]:
class DinoScreenCapture:
    """Handles PipeWire screen capture with dynamic cropping & preprocessing."""

    def __init__(self, crop_region: Optional[Tuple[int, int, int, int]] = None):
        self.crop_region = crop_region  # (x, y, w, h)
        self.frame = None
        self.latest_frame = None
        self.pipeline = None
        self.sink = None
        self.session = None
        self._target_size = (96, 96)

    def start_capture(self, fd: int, node_id: int, width: int, height: int):
        """Initialize GStreamer pipeline with PipeWire source."""
        # Initialize GStreamer if not already done
        Gst.init(None)
        
        pipeline_str = (
            f"pipewiresrc fd={fd} path={node_id} ! "
            "video/x-raw ! "
            "videoconvert ! "
            "video/x-raw, format=BGR ! "
            "appsink name=sink emit-signals=true sync=false max-buffers=1 drop=true"
        )

        self.pipeline = Gst.parse_launch(pipeline_str)
        self.sink = self.pipeline.get_by_name("sink")
        self.sink.connect("new-sample", self._on_new_sample)
        self.pipeline.set_state(Gst.State.PLAYING)

    def _on_new_sample(self, sink):
        """Callback: convert GStreamer buffer → OpenCV frame."""
        sample = sink.emit("pull-sample")
        if sample:
            buffer = sample.get_buffer()
            caps = sample.get_caps()
            structure = caps.get_structure(0)
            width = structure.get_value("width")
            height = structure.get_value("height")

            success, map_info = buffer.map(Gst.MapFlags.READ)
            if success:
                try:
                    # Create numpy array from buffer data
                    frame = np.ndarray(
                        (height, width, 3), buffer=map_info.data, dtype=np.uint8
                    ).copy()
                    self.latest_frame = frame
                finally:
                    buffer.unmap(map_info)

                return Gst.FlowReturn.OK
            return Gst.FlowReturn.ERROR
        return Gst.FlowReturn.ERROR

    def _resize_with_padding(self, frame: np.ndarray, target_size: Tuple[int, int]) -> np.ndarray:
        """
        Resize frame to target size maintaining aspect ratio with padding.
        
        Uses letterboxing technique - scales to fit within target dimensions
        then pads the remaining space with zeros (black).
        """
        target_h, target_w = target_size
        orig_h, orig_w = frame.shape[:2]
        
        # Calculate scaling factor to fit within target dimensions
        scale = min(target_w / orig_w, target_h / orig_h)
        
        # Calculate new dimensions
        new_w = int(orig_w * scale)
        new_h = int(orig_h * scale)
        
        # Resize frame
        resized = cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_AREA)
        
        # Create padded canvas
        if len(frame.shape) == 2:  # Grayscale
            padded = np.zeros((target_h, target_w), dtype=frame.dtype)
        else:  # Color
            padded = np.zeros((target_h, target_w, frame.shape[2]), dtype=frame.dtype)
        
        # Calculate padding offsets (center the image)
        y_offset = (target_h - new_h) // 2
        x_offset = (target_w - new_w) // 2
        
        # Place resized image in center of padded canvas
        padded[y_offset:y_offset + new_h, x_offset:x_offset + new_w] = resized
        
        return padded

    def get_frame(self, grayscale: bool = True) -> Optional[np.ndarray]:
        """Get latest preprocessed frame."""
        if self.latest_frame is None:
            return None
        
        frame = self.latest_frame.copy()
        
        # Apply crop region if specified
        if self.crop_region:
            x, y, w, h = self.crop_region
            # Ensure crop region is within bounds
            frame_h, frame_w = frame.shape[:2]
            x = max(0, min(x, frame_w))
            y = max(0, min(y, frame_h))
            w = min(w, frame_w - x)
            h = min(h, frame_h - y)
            frame = frame[y:y + h, x:x + w]
        
        # Convert to grayscale if requested
        if grayscale:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        # Resize to 84x84 with aspect ratio preservation and padding
        frame = self._resize_with_padding(frame, self._target_size)
        
        return frame

    def set_crop_region(self, region: Tuple[int, int, int, int]):
        """Set crop region for subsequent frames."""
        self.crop_region = region

    def stop(self):
        """Cleanup GStreamer pipeline."""
        if self.pipeline:
            self.pipeline.set_state(Gst.State.NULL)
            self.pipeline = None
        if self.session:
            self.session.close()
            self.session = None
        self.sink = None
        self.latest_frame = None

    def __enter__(self):
        """Context manager entry."""
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        """Context manager exit with cleanup."""
        self.stop()
        return False

In [3]:
# # == Testing ==

# import time
# crop = None

# print("[info]  Opening system screen picker... Select the Chrome Dino window.")
# portal = PortalCapture()
# session = portal.select_window()

# if not session:
#     print("[info]  No window selected.")
#     sys.exit(1)

# print(f"[info]  Stream acquired: {session.width}x{session.height}")

# capture = DinoScreenCapture(crop_region=crop)
# capture.start_capture(
#     fd=session.fd,
#     node_id=session.node_id,
#     width=session.width,
#     height=session.height
# )

# print("[info]  Capture started. Press 'q' to quit.")

# try:
#     prev_time = time.time()
#     while True:
#         # Get grayscale frame
#         frame = capture.get_frame(grayscale=True)
        
#         if frame is not None:
#             # Calculate FPS
#             current_time = time.time()
#             delta = current_time - prev_time
#             fps = 1 / delta if delta > 0 else 0
#             prev_time = current_time

#             # Convert back to BGR just for display (OpenCV needs color for imshow usually, 
#             # though it handles gray fine. Let's stack to 3 channels for visibility)
#             display_frame = cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)
            
#             # Add info text
#             h, w = frame.shape
#             cv2.putText(display_frame, f"Shape: {w}x{h} | FPS: {fps:.2f}", (10, 30), 
#                         cv2.FONT_HERSHEY_SIMPLEX, 0.3, (0, 255, 0), 1)
#             if crop:
#                 cv2.putText(display_frame, "Cropped", (10, 60), 
#                             cv2.FONT_HERSHEY_SIMPLEX, 0.3, (0, 0, 255), 1)
            
#             cv2.imshow("DinoScreenCapture Test", display_frame)

#         key = cv2.waitKey(1) & 0xFF
#         if key == ord('q'):
#             break
            
# except KeyboardInterrupt:
#     print("\n[error]  Interrupted")
# finally:
#     capture.stop()
#     session.close()
#     cv2.destroyAllWindows()
#     print("[info]  Cleanup complete")

---

# WS Server

run it in console
```js
(function () {
    const WS_SERVER = 'ws://localhost:8766';
    let ws = null;
    let reconnectTimeout = null;
    const PING_INTERVAL = 5_000; // 10 seconds
    let pingInterval = null;

    function connect() {
        ws = new WebSocket(WS_SERVER);

        ws.onopen = () => {
            console.log('[RemoteExec] Connected to WebSocket server');
            startPing();
        };

        ws.onmessage = async (event) => {
            try {
                const data = JSON.parse(event.data);

                if (data.type === 'pong') {
                    // Keep-alive response, nothing to do
                    return;
                }

                if (data.type === 'execute') {
                    const code = data.code;

                    // Wrap user code in an async function
                    let result;
                    try {
                        // Use new Function to avoid eval restrictions and enable async
                        const fn = new Function('return (async () => {' + code + '})()');
                        result = fn(); // This returns a Promise
                    } catch (e) {
                        // Syntax error or function creation failed
                        sendResult({
                            type: 'execution_result',
                            uid: data.uid,
                            result: null,
                            error: '[JS Parse Error] ' + (e.stack || e.toString()),
                        });
                        return;
                    }

                    // Handle Promise resolution
                    Promise.resolve(result)
                        .then(value => {
                            // Serialize result safely
                            try {
                                sendResult({
                                    type: 'execution_result',
                                    uid: data.uid,
                                    result: value,
                                    error: null,
                                });
                            } catch (e) {
                                sendResult({
                                    type: 'execution_result',
                                    uid: data.uid,
                                    result: null,
                                    error: '[Serialize Error] ' + (e.stack || e.toString()),
                                });
                            }
                        })
                        .catch(err => {
                            sendResult({
                                type: 'execution_result',
                                uid: data.uid,
                                error: String(err),
                                result: null,
                            });
                        });
                }
            } catch (e) {
                console.error('[RemoteExec] Error processing message:', e);
            }
        };

        ws.onclose = () => {
            console.log('[RemoteExec] Disconnected from WebSocket server');
            stopPing();
            scheduleReconnect();
        };

        ws.onerror = (error) => {
            console.error('[RemoteExec] WebSocket error:', error);
        };
    }

    function sendResult(payload) {
        if (ws && ws.readyState === WebSocket.OPEN) {
            ws.send(JSON.stringify(payload));
        }
    }

    function startPing() {
        stopPing();
        pingInterval = setInterval(() => {
            if (ws && ws.readyState === WebSocket.OPEN) {
                ws.send(JSON.stringify({
                    type: 'ping',
                    uid: crypto.randomUUID(),
                    timestamp: Date.now()
                }));
            }
        }, PING_INTERVAL);
    }

    function stopPing() {
        if (pingInterval) {
            clearInterval(pingInterval);
            pingInterval = null;
        }
    }

    function scheduleReconnect() {
        if (reconnectTimeout) {
            clearTimeout(reconnectTimeout);
        }
        reconnectTimeout = setTimeout(() => {
            console.log('[RemoteExec] Reconnecting...');
            connect();
        }, 3000);
    }

    // Start connection
    connect();
})();
```

In [4]:
class WebSocketServer:
    def __init__(self, host="localhost", port=8766):
        self.host = host
        self.port = port
        
        # Single client state
        self.client_ws = None
        self._connected = threading.Event()
        self._lock = threading.Lock()
        
        # Message handling
        self._msg_counter = 0
        self._pending_event = threading.Event()
        self._pending_id = None
        self._pending_result = None
        
        # Server state
        self.loop = None
        self._server_started = threading.Event()
        self._server = None  # Reference to the websocket server for graceful shutdown
        self._close_lock = threading.Lock()  # Prevent concurrent close attempts

    async def handle_client(self, websocket):
        # Register client
        with self._lock:
            # If replacing old client, cancel any pending request
            if self.client_ws is not None:
                print("Client reconnected, replacing old connection")
                self._cancel_pending()
            self.client_ws = websocket
            self._connected.set()
        
        print("Client connected", flush=True)

        try:
            async for message in websocket:
                try:
                    data = json.loads(message)
                    await self._handle_message(data)
                except json.JSONDecodeError:
                    pass  # Ignore invalid JSON
        except websockets.exceptions.ConnectionClosed:
            pass
        finally:
            with self._lock:
                if self.client_ws is websocket:
                    self.client_ws = None
                    self._connected.clear()
                    self._cancel_pending()
            print("Client disconnected")

    def _cancel_pending(self):
        """Cancel pending request on disconnect/reconnect"""
        if self._pending_id is not None:
            self._pending_result = {"error": "Client disconnected", "result": None}
            self._pending_event.set()

    async def _handle_message(self, data):
        """Minimal validation, fast path for execution_result"""
        msg_type = data.get('type')
        
        if msg_type == 'pong':
            return
            
        if msg_type == 'execution_result':
            msg_id = data.get('uid')
            with self._lock:
                if msg_id == self._pending_id and self._pending_id is not None:
                    self._pending_result = data
                    self._pending_event.set()

    async def _send_execute(self, code):
        """Async send - runs in server loop"""
        with self._lock:
            if self.client_ws is None:
                raise RuntimeError("No client connected")
            
            self._msg_counter += 1
            msg_id = self._msg_counter
            
            # Reset pending state
            self._pending_event.clear()
            self._pending_id = msg_id
            self._pending_result = None
            
            # Send directly to websocket - let it raise if closed
            try:
                await self.client_ws.send(json.dumps({
                    "type": "execute",
                    "uid": msg_id,
                    "code": code
                }))
            except websockets.exceptions.ConnectionClosed:
                raise RuntimeError("Client disconnected during send")
            
            return msg_id

    def execute_js(self, code):
        """
        Synchronous blocking call from main thread.
        Waits indefinitely until response received.
        """
        # Wait for server to be ready
        self._server_started.wait()
        
        # Wait for client connection
        if not self._connected.wait(timeout=30):
            raise RuntimeError("No client connected within 30s")
        
        # Submit to server loop
        future = asyncio.run_coroutine_threadsafe(
            self._send_execute(code), 
            self.loop
        )
        
        try:
            msg_id = future.result(timeout=5)  # Should be instant
        except Exception as e:
            raise RuntimeError(f"Failed to send: {e}")
        
        # Wait for response (blocking, no timeout)
        self._pending_event.wait()
        
        with self._lock:
            result = self._pending_result
            self._pending_id = None
            self._pending_result = None
        
        if result.get('error'):
            raise RuntimeError(result['error'])
        return result.get('result')

    async def _close_async(self):
        """Internal async method to perform shutdown"""
        # Close client websocket if connected
        if self.client_ws is not None:
            try:
                await self.client_ws.close(code=1000, reason="Server shutting down")
            except Exception:
                pass  # Ignore errors during force close
            self.client_ws = None
        
        # Cancel pending requests
        self._cancel_pending()
        self._connected.clear()
        
        # Close server socket
        if self._server is not None:
            self._server.close()
            try:
                await asyncio.wait_for(self._server.wait_closed(), timeout=2.0)
            except asyncio.TimeoutError:
                pass  # Force continue even if wait_closed hangs
            self._server = None
        
        self._server_started.clear()

    def _force_cleanup(self):
        """Force release resources when graceful shutdown fails"""
        with self._lock:
            # Nullify references to allow GC to close sockets
            self.client_ws = None
            self._server = None
            self.loop = None
            
            # Reset all threading primitives
            self._connected.clear()
            self._server_started.clear()
            self._pending_event.set()  # Unblock any waiting threads
            self._pending_id = None
            self._pending_result = {"error": "Server closed", "result": None}
        
        print("Server resources force-released")

    def close(self, timeout=5):
        """
        Synchronous blocking call to close the server.
        Attempts graceful shutdown first, then force-cleans if needed.
        Ensures bound address is released even on failure.
        
        Args:
            timeout: Maximum time to wait for graceful shutdown (seconds)
        
        Raises:
            RuntimeError: If close fails unexpectedly
        """
        # Prevent concurrent close attempts using lock
        if not self._close_lock.acquire(blocking=False):
            return  # Already closing from another thread
        
        try:
            # Quick path: server never started
            if self.loop is None or not self._server_started.is_set():
                self._force_cleanup()
                return
            
            # Try graceful async shutdown
            if self.loop.is_running() and not self.loop.is_closed():
                future = asyncio.run_coroutine_threadsafe(
                    self._close_async(),
                    self.loop
                )
                try:
                    future.result(timeout=timeout)
                    return  # Success
                except (asyncio.TimeoutError, RuntimeError, Exception):
                    pass  # Fall through to force cleanup
            
            # Fallback: force cleanup to ensure address release
            self._force_cleanup()
            
        finally:
            self._close_lock.release()

    def start_server(self):
        async def run():
            self.loop = asyncio.get_event_loop()
            self._server = await websockets.serve(
                self.handle_client, 
                self.host, 
                self.port,
                ping_interval=None,  # Disable auto-ping, client handles it
                ping_timeout=None
            )
            print(f"Server started on ws://{self.host}:{self.port}")
            self._server_started.set()
            await self._server.wait_closed()

        asyncio.run(run())

In [5]:
# # == Testing ==
# server = WebSocketServer()
# server_thread = threading.Thread(target=server.start_server, daemon=True)
# server_thread.start()
# time.sleep(3)
# # Blocks until response (no timeout)
# server.execute_js("return 2 + 58")
# server.close()

In [6]:
class Controller:
    def __init__(self, server):
        self.server = server
        self.is_ducking = False

    def _keyev(self, ev_type: str, key: str) -> str:
        assert ev_type in ["keyup", "keydown"]
        assert key in ["ArrowUp", "ArrowDown"]
        code = {"ArrowUp": 38, "ArrowDown": 40}[key]

        return f"""
            document.body.dispatchEvent(new KeyboardEvent('{ev_type}', {{
                key: "{key}",
                code: "{key}",
                bubbles: true,
                cancelable: true,
                view: window,
                which: {code},
                keyCode: {code},
            }}))
        """

    def reset(self):
        self.is_ducking = False
        return self.server.execute_js("""
            if (runnerInstance.playing) runnerInstance.stop()
            runnerInstance.restart()

            const rect = document.querySelector('.runner-canvas').getBoundingClientRect()
            return [rect.x, rect.y + (window.outerHeight - window.innerHeight), rect.width, rect.height]
        """)

    def get_status(self):
        return self.server.execute_js("""
            return {
                status: runnerInstance.crashed ? "crashed" : (runnerInstance.paused ? "paused" : "playing"),
                distance: parseInt(runnerInstance.distanceMeter.digits.join("") || '0')
            }
        """)

    def jump(self):
        ducking_flag = int(self.is_ducking)
        self.server.execute_js(f"""
            if ({ducking_flag})
                requestAnimationFrame(() => {self._keyev('keyup', 'ArrowDown')});
            requestAnimationFrame(() => {self._keyev('keydown', 'ArrowUp')});
            requestAnimationFrame(() => {self._keyev('keyup', 'ArrowUp')});
            return true
        """)
        self.is_ducking = False

    def duck(self):
        not_ducking = int(not self.is_ducking)
        self.server.execute_js(f"""
            if ({not_ducking})
                requestAnimationFrame(() => {self._keyev('keydown', 'ArrowDown')});
            return true
        """)
        self.is_ducking = True

    def noop(self):
        ducking_flag = int(self.is_ducking)
        self.server.execute_js(f"""
            if ({ducking_flag})
                requestAnimationFrame(() => {self._keyev('keyup', 'ArrowDown')});
            return true
        """)
        self.is_ducking = False

In [7]:
# == Testing ==
# ctrl = Controller(server)
# ctrl.jump()
# ctrl.duck()
# ctrl.noop()
# ctrl.reset()
# ctrl.get_status()

In [8]:
# # == Testing ==
# print("[info]  Opening system screen picker... Select the Chrome Dino window.")
# portal = PortalCapture()
# session = portal.select_window()

# if not session:
#     print("[info]  No window selected.")
#     sys.exit(1)

# print(f"[info]  Stream acquired: {session.width}x{session.height}")


# crop = Controller.reset()
# crop[1] -= 15 # idk why the crop above isnt working, figure otu later

# capture = DinoScreenCapture(crop_region=crop)
# capture.start_capture(
#     fd=session.fd,
#     node_id=session.node_id,
#     width=session.width,
#     height=session.height
# )

# print("[info]  Capture started. Press 'q' to quit.")

# try:
#     prev_time = time.time()
#     while True:
#         # Get grayscale frame
#         start_time = time.perf_counter()
    
#         state = Controller.get_status()
#         frame = capture.get_frame(grayscale=True)
        
#         print(f"\rElapsed time: {time.perf_counter() - start_time:.4f}s", end="\r")


#         if frame is not None:
#             # Calculate FPS
#             current_time = time.time()
#             delta = current_time - prev_time
#             fps = 1 / delta if delta > 0 else 0
#             prev_time = current_time

#             # Convert back to BGR just for display (OpenCV needs color for imshow usually, 
#             # though it handles gray fine. Let's stack to 3 channels for visibility)
#             display_frame = cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)
            
#             # Add info text
#             h, w = frame.shape
#             cv2.putText(display_frame, f"Shape: {w}x{h} | FPS: {fps:.2f} | State: {state['status']} | {state['distance']}", (10, 30), 
#                         cv2.FONT_HERSHEY_SIMPLEX, 0.1, (0, 255, 0), 1)
#             if crop:
#                 cv2.putText(display_frame, "Cropped", (10, 60), 
#                             cv2.FONT_HERSHEY_SIMPLEX, 0.1, (0, 0, 255), 1)
            
#             cv2.imshow("DinoScreenCapture Test", display_frame)

#         key = cv2.waitKey(1) & 0xFF
#         if key == ord('q'):
#             break

#         if key == ord('r'):
#             Controller.reset()

#         if key == 82: # ArrowUp
#             Controller.jump()

#         if key == 84: #  ArrowDown
#             Controller.duck()

#         if key == 83: #  ArrowRight
#             Controller.noop()

# except KeyboardInterrupt:
#     print("\n[error]  Interrupted")
# finally:
#     capture.stop()
#     session.close()
#     cv2.destroyAllWindows()
#     print("[info]  Cleanup complete")

---

---

---

In [9]:
class DinoEnv(gym.Env):
    metadata = {"render_modes": ["human"]}
    
    def __init__(
        self,
        host="localhost", port=8766,
        obs_shape=(96, 96, 1),      # final CNN-ready shape
        target_width=84,             # resize target for larger dimension
        reward_type="dense",         # "dense" or "sparse"
        frame_skip=1,
        max_episode_steps=1000,
        render_mode=None,
    ):
        super().__init__()
        
        # 1️⃣ Start WebSocket server (daemon thread)
        self.server = WebSocketServer(host, port)
        self.server_thread = threading.Thread(
            target=self.server.start_server, daemon=True
        )
        self.server_thread.start()
        # Wait ~2s for server to bind
        time.sleep(2)

        # 1️⃣ Start Controller
        self.controller = Controller(self.server)
        
        
        # 2️⃣ Defer PipeWire capture init to first reset() 
        #    (avoids blocking __init__ if user isn't ready to select window)
        self.capture = DinoScreenCapture()
        self.session = None  # PortalSession, set in _init_capture()
        
        # 3️⃣ Spaces
        self.observation_space = spaces.Box(
            low=0, high=255, shape=obs_shape, dtype=np.uint8
        )
        self.action_space = spaces.Discrete(3)  # 0:noop, 1:jump, 2:duck
        
        # 4️⃣ Config
        self.target_width = target_width
        self.reward_type = reward_type
        self.frame_skip = frame_skip
        self.max_episode_steps = max_episode_steps
        self.render_mode = render_mode
        
        # 5️⃣ Runtime state
        self.prev_distance = 0
        self.step_count = 0
        self._initialized_capture = False

    def _init_capture(self):
        """One-time setup: select window + start GStreamer pipeline."""
        print("[env] Select the Chrome Dino window in the system picker...")
        portal = PortalCapture()
        self.session = portal.select_window()  # BLOCKS for user interaction
        if not self.session:
            raise RuntimeError("Window selection cancelled or failed")
            
        self.capture.start_capture(
            fd=self.session.fd,
            node_id=self.session.node_id,
            width=self.session.width,
            height=self.session.height
        )
        self._initialized_capture = True
        print(f"[env] Capture started: {self.session.width}x{self.session.height}")

    def _preprocess(self, frame: np.ndarray) -> np.ndarray:
        """Resize 552x150 → preserve aspect → zero-pad to 84x84x1."""
        # frame shape: (150, 552) grayscale
        h, w = frame.shape
        
        # Scale so that width == target_width (84), compute new height
        scale = self.target_width / w
        new_h = int(h * scale)  # ~23 for 150*(84/552)
        
        # Resize
        resized = cv2.resize(frame, (self.target_width, new_h), 
                            interpolation=cv2.INTER_AREA)
        
        # Create 84x84 canvas, paste centered
        canvas = np.zeros((self.target_width, self.target_width), dtype=np.uint8)
        y_offset = (self.target_width - new_h) // 2
        canvas[y_offset:y_offset+new_h, :] = resized
        
        # Add channel dim → (84,84,1)
        return np.expand_dims(canvas, axis=-1)

    def _get_obs(self) -> np.ndarray:
        """Fetch latest frame + preprocess."""
        raw = self.capture.get_frame(grayscale=True)
        if raw is None:
            return np.zeros(self.observation_space.shape, dtype=np.uint8)
        return self._preprocess(raw)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        
        # Lazy-init capture on first reset
        if not self._initialized_capture:
            self._init_capture()
            
        # Reset game via JS → get canvas bbox for cropping
        crop = self.server.execute_js("""
            if (runnerInstance.playing) runnerInstance.stop();
            runnerInstance.restart();
            const r = document.querySelector('.runner-canvas').getBoundingClientRect();
            // Return [x, y, w, h]; adjust y for window chrome if needed
            return [r.x, r.y + (window.outerHeight - window.innerHeight), r.width, r.height];
        """)
        # Optional: fine-tune crop (your -15px fix)
        crop[1] -= 15
        self.capture.set_crop_region(tuple(crop))
        
        # Let game stabilize (skip first few frames)
        for _ in range(5):
            self.controller.get_status()
            time.sleep(0.01)
            
        obs = self._get_obs()
        self.prev_distance = 0
        self.step_count = 0
        return obs, {}

    def step(self, action: int):
        # 1️⃣ Execute action
        if action == 0:
            self.controller.noop()
        elif action == 1:
            self.controller.jump()
        elif action == 2:
            self.controller.duck()
        # else: ignore invalid

        total_reward = 0.0
        done = False
        info = {}

        # 2️⃣ Frame-skip loop
        for _ in range(self.frame_skip):
            state = self.controller.get_status()  # {'status':..., 'distance':...}
            dist = state['distance']
            
            # Reward calculation
            if self.reward_type == "dense":
                step_reward = dist - self.prev_distance
            else:  # sparse
                step_reward = 1 if state['status'] != 'crashed' else 0
                
            total_reward += step_reward
            self.prev_distance = dist
            
            if state['status'] == 'crashed':
                total_reward -= 10  # crash penalty
                done = True
                info['reason'] = 'crashed'
                break
                
        # 3️⃣ Get observation
        obs = self._get_obs()
        self.step_count += 1
        
        # 4️⃣ Termination checks
        if self.step_count >= self.max_episode_steps:
            done = True
            info['reason'] = 'max_steps'
            
        info.update({
            'distance': self.prev_distance,
            'status': state['status'],
            'episode_steps': self.step_count
        })
        
        # Gymnasium API: return 5 values
        return obs, total_reward, done, False, info

    def render(self):
        if self.render_mode != "human":
            return
        # Show preprocessed obs (optional: show raw capture in separate window)
        obs = self._get_obs()
        display = cv2.cvtColor(obs.squeeze(), cv2.COLOR_GRAY2BGR)
        cv2.imshow("DinoEnv Observation", display)
        cv2.waitKey(1)

    def close(self):
        if self.capture:
            self.capture.stop()
            self.capture = None
        if self.session:
            self.session.close()
            self.session = None
        if self.server:
            self.server.close()
        cv2.destroyAllWindows()

In [10]:
## TESTING ##
env = DinoEnv(frame_skip=2, reward_type="dense", render_mode="human")

obs, info = env.reset()
print(f"Reset done. Obs shape: {obs.shape}")

for ep in range(3):
    done = False
    ep_reward = 0
    while not done:
        # Random policy for testing
        action = env.action_space.sample()
        obs, reward, done, truncated, info = env.step(action)
        ep_reward += reward
        print(f"\r[Ep {ep+1}] Step {info['episode_steps']} | "
              f"Dist: {info['distance']:4d} | "
              f"Status: {info['status']:7s} | "
              f"Reward: {reward:+.1f} | "
              f"Total: {ep_reward:+.1f}", end="")
        env.render()  # optional
    print(f"\n[Ep {ep+1}] Finished. Final distance: {info['distance']}")
    if not done:  # crashed, so reset
        obs, info = env.reset()

env.close()

Server started on ws://localhost:8766
[env] Select the Chrome Dino window in the system picker...
Client connected
[env] Capture started: 0x0
Reset done. Obs shape: (84, 84, 1)
[Ep 1] Step 21 | Dist:    0 | Status: playing | Reward: +0.0 | Total: +0.0

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in "/home/ezio4df/projects/play-TRexRunner/.venv/lib/python3.13/site-packages/cv2/qt/plugins"


[Ep 1] Step 634 | Dist:   44 | Status: crashed | Reward: -10.0 | Total: +34.0
[Ep 1] Finished. Final distance: 44
[Ep 2] Step 635 | Dist:   44 | Status: crashed | Reward: -10.0 | Total: -10.0
[Ep 2] Finished. Final distance: 44
[Ep 3] Step 636 | Dist:   44 | Status: crashed | Reward: -10.0 | Total: -10.0
[Ep 3] Finished. Final distance: 44
Client disconnected
